# Espectrograma sencillo con Matplotlib

**UNIE_VozClon** · Reinicia el kernel y ejecuta las tres celdas en orden.
La última abre una ventana de Matplotlib y activa el micrófono. Cierra esa ventana
o pulsa **Interrumpir** para detener y liberar la entrada.

Recorrido: elegir micrófono → configurar → leer audio y actualizar el gráfico.
¿Qué cambia al sostener una vocal, subir su altura o pronunciar «ssss»?

Solo entradas WASAPI. El audio permanece en una ventana móvil en memoria; no se guarda a disco.
La FFT usa las muestras de un frame completo, sin zero-padding. El color es PSD en
**dB re 1 FS²/Hz**, con escala fija; no son dB SPL. El gris indica datos aún no disponibles
o pérdida de muestras. Los controles y el código son independientes de la versión estable.

La captura y la ventana Tk reales quedan pendientes de tu ejecución. No se instala nada.


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd
from scipy.signal import spectrogram, get_window

get_ipython().run_line_magic("matplotlib", "tk")

api = next(a for a in sd.query_hostapis() if a["name"] == "Windows WASAPI")
for numero in api["devices"]:
    entrada = sd.query_devices(numero)
    if entrada["max_input_channels"] > 0:
        print(numero, entrada["name"])


9 Varios micrófonos (Realtek(R) Audio)


In [4]:
DISPOSITIVO = api["default_input_device"]  # O un número de la lista.
FS = None                 # None: frecuencia de muestreo predeterminada del micrófono.
VENTANA_MS = 32            # Duración de cada frame.
TIPO_VENTANA = "hann"      # Hann periódica; también admite "hamming" o "boxcar".
SOLAPAMIENTO = 0.75        # 75 %; el hop avanza el 25 % del frame.
SEGUNDOS = 6              # Audio conservado en la ventana móvil.
FRECUENCIA_MAX = 8000      # Hz visibles, hasta Nyquist.
FPS = 12                  # Objetivo de actualizaciones por segundo.
DB_MIN, DB_MAX = -110, -10 # Escala fija de color.


In [5]:
fs = int(sd.query_devices(DISPOSITIVO, "input")["default_samplerate"] if FS is None else FS)
if not (0 <= SOLAPAMIENTO < 1 and fs > 0 and FPS > 0 and VENTANA_MS > 0):
    raise ValueError("Revisa FS, FPS, VENTANA_MS y SOLAPAMIENTO.")
n_fft = max(16, 2 * round(fs * VENTANA_MS / 2000))
hop = max(1, int(n_fft * (1 - SOLAPAMIENTO)))
bloque = max(1, int(fs / FPS))
if SEGUNDOS * fs < max(n_fft, bloque):
    raise ValueError("SEGUNDOS debe contener al menos un frame y un bloque de lectura.")
ventana = np.hanning(n_fft + 1)[:-1] if TIPO_VENTANA == "hann" else get_window(TIPO_VENTANA, n_fft, fftbins=True)
audio = np.full(int(SEGUNDOS * fs), np.nan)
f, t, potencia = spectrogram(audio, fs, window=ventana, nperseg=n_fft,
                             noverlap=n_fft - hop, detrend=False, scaling="density")
visibles = f <= FRECUENCIA_MAX

plt.ion()
fig, ax = plt.subplots(figsize=(6.4, 3.6))
colores = plt.get_cmap("magma").copy()
colores.set_bad("#D9DDE2")
imagen = ax.imshow(potencia[visibles], origin="lower", aspect="auto", interpolation="nearest",
                   extent=(t[0] - hop / (2 * fs) - SEGUNDOS, t[-1] + hop / (2 * fs) - SEGUNDOS,
                           -fs / (2 * n_fft * 1000), (f[visibles][-1] + fs / (2 * n_fft)) / 1000),
                   cmap=colores, vmin=DB_MIN, vmax=DB_MAX)
ax.set(xlabel="Time (s)", ylabel="Frequency (kHz)",
       xlim=(-SEGUNDOS, 0), ylim=(0, min(FRECUENCIA_MAX, fs / 2) / 1000))
fig.colorbar(imagen, ax=ax, label="PSD (dB re 1 FS²/Hz)", extend="both")
fig.tight_layout()
print(f"fs={fs} Hz | FFT={n_fft} muestras | hop={hop} muestras")

try:
    plt.show(block=False)
    with sd.InputStream(device=DISPOSITIVO, samplerate=fs, channels=1,
                        dtype="float32", latency="high") as microfono:
        while plt.fignum_exists(fig.number):
            muestras, perdida = microfono.read(bloque)
            if perdida:
                audio[:] = np.nan
            audio = np.roll(audio, -bloque)
            audio[-bloque:] = muestras[:, 0]
            f, t, potencia = spectrogram(audio, fs, window=ventana, nperseg=n_fft,
                                         noverlap=n_fft - hop, detrend=False, scaling="density")
            db = 10 * np.log10(np.maximum(potencia[visibles], 1e-20))
            imagen.set_data(db)
            fig.canvas.draw_idle()
            plt.pause(0.001)
except KeyboardInterrupt:
    pass
finally:
    plt.close(fig)
    print("Micrófono liberado.")


fs=48000 Hz | FFT=1536 muestras | hop=384 muestras
Micrófono liberado.


## Análisis acústico

Al sostener una vocal, las cuerdas vocales producen una señal periódica. En el
espectrograma se observan armónicos como bandas horizontales definidas, con mayor
energía normalmente en las frecuencias bajas y medias (por debajo de unos 3–4 kHz).
La distribución de energía de la vocal también forma sus resonancias o formantes.

Al pronunciar una fricativa sorda como «ssss», el flujo turbulento genera ruido
aperiódico de banda ancha: no aparecen armónicos regulares y la energía se dispersa,
con frecuencia más marcada por encima de 4 kHz. La forma exacta depende de la voz,
el micrófono, la distancia y el nivel de ruido; la escala de este gráfico es PSD,
no una medida de dB SPL.


### Subir la altura (tono / pitch) y prueba en tiempo real
* **Al subir la altura**: aumenta la frecuencia fundamental ($f_0$) y la separación vertical entre las bandas de los armónicos (múltiplos enteros de $f_0$), mientras que las regiones de formantes asociadas al timbre vocálico mantienen su rango general.
* **Observación práctica con el micrófono**: [añade aquí qué notaste al sostener la vocal, subir el tono y hacer «ssss» frente a tu micrófono].